In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import (
    shapiro,
    pearsonr,
    spearmanr,
    mannwhitneyu,
    ttest_ind,
    ttest_rel,
    wilcoxon,
    chi2_contingency
)
MAIN_FOLDER = "../"
ANGULATION_FOLDER = "4-right_left_angulation_agreement-2/"

In [ ]:
# ============================================================
# 1. Load dataset
# ============================================================
FILE_NAME = "paired_list"

# Read the excel file
paired = pd.read_excel("../../"+FILE_NAME+".xlsx")

# Show the head
print(paired.tail())

In [ ]:
# Check whether left and right angulations are the same
paired["angulation_match"] = (
    paired["winter_angulation_left"] == paired["winter_angulation_right"]
)
paired.tail()

In [ ]:
total_bilateral_patients = len(paired)
matched_patients = paired["angulation_match"].sum()
unmatched_patients = total_bilateral_patients - matched_patients

match_percentage = matched_patients / total_bilateral_patients * 100
unmatch_percentage = unmatched_patients / total_bilateral_patients * 100

summary = pd.DataFrame({
    "Metric": [
        "Total bilateral patients",
        "Matched angulation patients",
        "Unmatched angulation patients",
        "Matched angulation percentage",
        "Unmatched angulation percentage"
    ],
    "Value": [
        total_bilateral_patients,
        matched_patients,
        unmatched_patients,
        round(match_percentage, 2),
        round(unmatch_percentage, 2)
    ]
})


In [ ]:
print("\nRight-left Winter angulation agreement summary")
summary

In [ ]:
print("\nExample paired angulation table")
paired.tail()

In [ ]:
# Cross-tabulation between left and right Winter angulation
angulation_crosstab = pd.crosstab(
    paired["winter_angulation_left"],
    paired["winter_angulation_right"]
)

print("\nLeft vs Right Winter Angulation Cross-tabulation")
angulation_crosstab

In [ ]:
# Percentage version by left angulation
angulation_crosstab_percent = pd.crosstab(
    paired["winter_angulation_left"],
    paired["winter_angulation_right"],
    normalize="index"
) * 100

print("\nLeft vs Right Winter Angulation Cross-tabulation (%)")
angulation_crosstab_percent.round(2)

In [ ]:
# Create output folder
output_folder = MAIN_FOLDER+ANGULATION_FOLDER
os.makedirs(output_folder, exist_ok=True)

paired.to_excel(
    os.path.join(output_folder, "paired_left_right_angulation.xlsx"),
    index=False
)

summary.to_excel(
    os.path.join(output_folder, "right_left_angulation_agreement_summary.xlsx"),
    index=False
)

angulation_crosstab.to_excel(
    os.path.join(output_folder, "left_right_angulation_crosstab.xlsx")
)

angulation_crosstab_percent.round(2).to_excel(
    os.path.join(output_folder, "left_right_angulation_crosstab_percent.xlsx")
)

In [ ]:
agreement_counts = paired["angulation_match"].value_counts()

agreement_counts.index = agreement_counts.index.map({
    True: "Matched",
    False: "Unmatched"
})

def my_autopct(pct):
    return ("%1.1f%%" % pct) if pct > 1.0 else ""


# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(8, 8))

colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(agreement_counts)))

#colors = plt.cm.Pastel1(np.linspace(0, 1, len(agreement_counts)))

wedges, texts, autotexts = plt.pie(
    agreement_counts,
    autopct=my_autopct,
    startangle=90,
    pctdistance=0.85,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w")
)

# Set percentage text style
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(12)
    autotext.set_fontweight("bold")


plt.title("Right-Left Winter Angulation Agreement")

plt.legend(
    wedges,
    agreement_counts.index.astype(str),
    title="Agreement Status",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

plt.tight_layout()
plt.savefig(
    os.path.join(output_folder, "right_left_angulation_agreement_donut_chart.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import matplotlib.colors as mcolors
# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 15,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

# ---------------------------
# Create a darker blue colormap
# ---------------------------
base_cmap = plt.cm.Blues

# Use only the medium-to-dark part of the Blues colormap
# 0.25 avoids very pale blue/white colors
# 0.90 avoids extremely dark colors
blue_colors = base_cmap(np.linspace(0.15, 0.95, 256))

custom_blues = mcolors.LinearSegmentedColormap.from_list(
    "custom_blues",
    blue_colors
)

# ---------------------------
# Figure
# ---------------------------
fig, ax = plt.subplots(figsize=(8, 6))

img = ax.imshow(
    angulation_crosstab,
    aspect="auto",
    cmap=custom_blues
)

ax.set_title("(a) Winter angulation", pad=12)
ax.set_xlabel("Right side", labelpad=10)
ax.set_ylabel("Left side", labelpad=10)

ax.set_xticks(np.arange(len(angulation_crosstab.columns)))
ax.set_xticklabels(
    angulation_crosstab.columns,
    rotation=0,
    ha="center"
)

ax.set_yticks(np.arange(len(angulation_crosstab.index)))
ax.set_yticklabels(angulation_crosstab.index)

# ---------------------------
# Add counts inside cells
# ---------------------------

# Threshold controls when text becomes white.
threshold = angulation_crosstab.values.max() * 0.45

for i in range(len(angulation_crosstab.index)):
    for j in range(len(angulation_crosstab.columns)):
        value = angulation_crosstab.iloc[i, j]

        # Use white text on darker cells, dark text on lighter cells
        text_color = "white" if value >= threshold else "#222222"

        ax.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            color=text_color,
            fontsize=15,
            fontweight="bold"
        )

# ---------------------------
# Colorbar
# ---------------------------
cbar = plt.colorbar(img, ax=ax)
cbar.set_label("Number of Patients")

plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "left_right_angulation_crosstab_heatmap.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()

In [ ]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(
    paired["winter_angulation_left"],
    paired["winter_angulation_right"]
)

print("Cohen's Kappa for right-left Winter angulation agreement:", round(kappa, 4))

In [ ]:
######## OLD

In [ ]:
# ---------------------------
# Global style settings for academic appearance
# ---------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(8, 6))

plt.imshow(angulation_crosstab, aspect="auto", cmap="Blues")
#plt.imshow(angulation_crosstab, aspect="auto", cmap=colors)


plt.title("Left vs Right Winter Angulation Cross-tabulation")
plt.xlabel("Right Winter Angulation")
plt.ylabel("Left Winter Angulation")

plt.xticks(
    ticks=np.arange(len(angulation_crosstab.columns)),
    labels=angulation_crosstab.columns,
    rotation=30,
    ha="right"
)

plt.yticks(
    ticks=np.arange(len(angulation_crosstab.index)),
    labels=angulation_crosstab.index
)

# Add counts inside cells
for i in range(len(angulation_crosstab.index)):
    for j in range(len(angulation_crosstab.columns)):
        value = angulation_crosstab.iloc[i, j]
        plt.text(j, i, str(value), ha="center", va="center")

plt.colorbar(label="Number of Patients")
plt.tight_layout()

plt.savefig(
    os.path.join(output_folder, "OLD.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()